In [1]:
"""
01_data_preprocessing.ipynb
Phase 2: Data Preprocessing Pipeline
Author: [Your Name]
Date: 2026-08-27
Description: Load, clean, and prepare Reddit and X datasets for analysis
"""

# Import libraries
%pip install --upgrade --quiet langdetect

import pandas as pd
import numpy as np
import json
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# NLP libraries
%pip install nltk -q
%pip install langdetect -q

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from langdetect import detect

# Utilities
from tqdm import tqdm
import sys
from pathlib import Path

# Add src to path
sys.path.append(str(Path.cwd().parent / 'src'))
from config import DATA_RAW, PROJECT_ROOT, DATA_PROCESSED, EVENT_START, TIME_WINDOW_HOURS

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

print("All imports loaded successfully!")
print(f"Data raw path: {DATA_RAW}")
print(f"Data processed path: {DATA_PROCESSED}")


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
All imports loaded successfully!
Data raw path: c:\Users\GALLOP SEA\Desktop\The_Echo_Of_Anonimity-1\src\data\raw
Data processed path: c:\Users\GALLOP SEA\Desktop\The_Echo_Of_Anonimity-1\src\data\processed


[nltk_data] Downloading package punkt to C:\Users\GALLOP
[nltk_data]     SEA\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\GALLOP
[nltk_data]     SEA\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\GALLOP
[nltk_data]     SEA\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
# Load Reddit data
reddit_file = PROJECT_ROOT / "data" / "raw" / 'reddit_comments.json'

with open(reddit_file, 'r') as f:
    # Load Reddit data
    reddit_file = PROJECT_ROOT / "data" / "raw" / 'reddit_comments.json'

    # Try common encodings for JSON files; Windows often defaults to cp1252
    for encoding in ('utf-8-sig', 'utf-8', 'cp1252', 'latin-1'):
        try:
            with open(reddit_file, 'r', encoding=encoding) as f:
                reddit_data = json.load(f)
            break
        except (UnicodeDecodeError, json.JSONDecodeError):
            continue
    else:
        raise ValueError(f"Could not decode Reddit JSON file: {reddit_file}")

    # Convert to DataFrame
    reddit_df = pd.DataFrame(reddit_data)

    print("=== REDDIT DATA ===")
    print(f"Shape: {reddit_df.shape}")
    print(f"Columns: {reddit_df.columns.tolist()}")
    print(f"\nData types:\n{reddit_df.dtypes}")
    print(f"\nFirst 5 rows:")
    display(reddit_df.head())

    print(f"\nMissing values:")
    print(reddit_df.isnull().sum())

    # Sample one row to understand structure
    print(f"\nSample row (first comment):")
    display(reddit_df.iloc[0])

# Convert to DataFrame
reddit_df = pd.DataFrame(reddit_data)

print("=== REDDIT DATA ===")
print(f"Shape: {reddit_df.shape}")
print(f"Columns: {reddit_df.columns.tolist()}")
print(f"\nData types:\n{reddit_df.dtypes}")
print(f"\nFirst 5 rows:")
display(reddit_df.head())

print(f"\nMissing values:")
print(reddit_df.isnull().sum())

# Sample one row to understand structure
print(f"\nSample row (first comment):")
display(reddit_df.iloc[0])

=== REDDIT DATA ===
Shape: (1000, 20)
Columns: ['subreddit', 'id', 'text', 'author', 'timestamp', 'submission_id', 'controversial', 'score', 'ups', 'downs', 'parent_id', 'datetime', 'title', 'upvote_ratio', 'upvotes', 'num_comments', 'comments_df', 'freepalestine_islamophobia', 'zionism_antisemitism', 'comment_df']

Data types:
subreddit                         str
id                                str
text                              str
author                            str
timestamp                         str
submission_id                     str
controversial                 float64
score                           int64
ups                           float64
downs                         float64
parent_id                         str
datetime                          str
title                             str
upvote_ratio                  float64
upvotes                       float64
num_comments                  float64
comments_df                    object
freepalestine_islamophob

,subreddit,id,text,author,timestamp,submission_id,controversial,score,ups,downs,parent_id,datetime,title,upvote_ratio,upvotes,num_comments,comments_df,freepalestine_islamophobia,zionism_antisemitism,comment_df
0,IsraelPalestine,k4qq7y8,source? trust me dude?,njojubcb,1697222508.0,1775k8e,0.0,1,1.0,0.0,1775k8e,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,worldnews,k3vtjkj,you will not convince me or any other rational...,Jsjtijobups,1696702548.0,17251fb,0.0,1,1.0,0.0,17251fb,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,politics,k7gjm7b,213 voted to keep him179 voted to expel him19 ...,sfgsjfej,1698893962.0,17lhfyi,0.0,12,12.0,0.0,k7g0gfb,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,islam,kakjfjn,if you have wronged him then you have to seek ...,ojijmptpqijtu,1700837432.0,182sgwc,0.0,1,1.0,0.0,182sgwc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,VaushV,k4k9zky,the question is false because palestinians are...,Pvudbtu_Efwjm,1697119334.0,1760w5y,0.0,1,1.0,0.0,k4k90n1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Missing values:
subreddit                       0
id                              0
text                            0
author                          0
timestamp                       0
submission_id                 500
controversial                 500
score                           0
ups                           500
downs                         500
parent_id                     500
datetime                      500
title                         500
upvote_ratio                  500
upvotes                       500
num_comments                  500
comments_df                   500
freepalestine_islamophobia    500
zionism_antisemitism          500
comment_df                    500
dtype: int64

Sample row (first comment):


subreddit                            IsraelPalestine
id                                           k4qq7y8
text                          source? trust me dude?
author                                      njojubcb
timestamp                               1697222508.0
submission_id                                1775k8e
controversial                                    0.0
score                                              1
ups                                              1.0
downs                                            0.0
parent_id                                    1775k8e
datetime                                         NaN
title                                            NaN
upvote_ratio                                     NaN
upvotes                                          NaN
num_comments                                     NaN
comments_df                                      NaN
freepalestine_islamophobia                       NaN
zionism_antisemitism                          

=== REDDIT DATA ===
Shape: (1000, 20)
Columns: ['subreddit', 'id', 'text', 'author', 'timestamp', 'submission_id', 'controversial', 'score', 'ups', 'downs', 'parent_id', 'datetime', 'title', 'upvote_ratio', 'upvotes', 'num_comments', 'comments_df', 'freepalestine_islamophobia', 'zionism_antisemitism', 'comment_df']

Data types:
subreddit                         str
id                                str
text                              str
author                            str
timestamp                         str
submission_id                     str
controversial                 float64
score                           int64
ups                           float64
downs                         float64
parent_id                         str
datetime                          str
title                             str
upvote_ratio                  float64
upvotes                       float64
num_comments                  float64
comments_df                    object
freepalestine_islamophob

,subreddit,id,text,author,timestamp,submission_id,controversial,score,ups,downs,parent_id,datetime,title,upvote_ratio,upvotes,num_comments,comments_df,freepalestine_islamophobia,zionism_antisemitism,comment_df
0,IsraelPalestine,k4qq7y8,source? trust me dude?,njojubcb,1697222508.0,1775k8e,0.0,1,1.0,0.0,1775k8e,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,worldnews,k3vtjkj,you will not convince me or any other rational...,Jsjtijobups,1696702548.0,17251fb,0.0,1,1.0,0.0,17251fb,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,politics,k7gjm7b,213 voted to keep him179 voted to expel him19 ...,sfgsjfej,1698893962.0,17lhfyi,0.0,12,12.0,0.0,k7g0gfb,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,islam,kakjfjn,if you have wronged him then you have to seek ...,ojijmptpqijtu,1700837432.0,182sgwc,0.0,1,1.0,0.0,182sgwc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,VaushV,k4k9zky,the question is false because palestinians are...,Pvudbtu_Efwjm,1697119334.0,1760w5y,0.0,1,1.0,0.0,k4k90n1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Missing values:
subreddit                       0
id                              0
text                            0
author                          0
timestamp                       0
submission_id                 500
controversial                 500
score                           0
ups                           500
downs                         500
parent_id                     500
datetime                      500
title                         500
upvote_ratio                  500
upvotes                       500
num_comments                  500
comments_df                   500
freepalestine_islamophobia    500
zionism_antisemitism          500
comment_df                    500
dtype: int64

Sample row (first comment):


subreddit                            IsraelPalestine
id                                           k4qq7y8
text                          source? trust me dude?
author                                      njojubcb
timestamp                               1697222508.0
submission_id                                1775k8e
controversial                                    0.0
score                                              1
ups                                              1.0
downs                                            0.0
parent_id                                    1775k8e
datetime                                         NaN
title                                            NaN
upvote_ratio                                     NaN
upvotes                                          NaN
num_comments                                     NaN
comments_df                                      NaN
freepalestine_islamophobia                       NaN
zionism_antisemitism                          

In [3]:
# Load X data
x_file = DATA_RAW / 'x_twitter_dataset.csv'

# Try different encodings if the first fails
try:
    x_df = pd.read_csv(x_file)
except UnicodeDecodeError:
    x_df = pd.read_csv(x_file, encoding='latin-1')
except:
    x_df = pd.read_csv(x_file, encoding='utf-8-sig')

print("=== X (TWITTER) DATA ===")
print(f"Shape: {x_df.shape}")
print(f"Columns: {x_df.columns.tolist()}")
print(f"\nData types:\n{x_df.dtypes}")
print(f"\nFirst 5 rows:")
display(x_df.head())

print(f"\nMissing values:")
print(x_df.isnull().sum())

# Sample one row
print(f"\nSample row:")
display(x_df.iloc[0])

=== X (TWITTER) DATA ===
Shape: (7323, 10)
Columns: ['number', 'link', 'text', 'user', 'likes', 'quotes', 'retweets', 'comments', 'label', 'cleaned_text']

Data types:
number          int64
link              str
text              str
user              str
likes           int64
quotes          int64
retweets        int64
comments        int64
label           int64
cleaned_text      str
dtype: object

First 5 rows:


,number,link,text,user,likes,quotes,retweets,comments,label,cleaned_text
0,1,https://twitter.com/BabylonBulletin/status/171...,🚨 #BREAKING 🚨 A high-ranking member of Israe...,Babylon Bulletin,2,0,0,0,0,breaking highranking member israeli special fo...
1,780,https://twitter.com/bahabreaking/status/178496...,Hamas: No ceasefire if Israel fails to meet de...,baha,0,0,0,0,0,hamas ceasefire israel fails meet demands hama...
2,781,https://twitter.com/ttimagazine/status/1784966...,Hamas Says ‘No Major Issues’ With Israeli Ceas...,The Truth International,0,0,0,0,0,hamas says major issues israeli ceasefire prop...
3,782,https://twitter.com/leapp77/status/17849657885...,UK PM urges Hamas to accept 40-day #ceasefire ...,Fa. pan,0,0,0,0,0,uk pm urges hamas accept 40day ceasefire hamas...
4,784,https://twitter.com/LeilaniAthenai1/status/178...,"It's not about #Zionism VS pro-#Palestine, but...",Leilani Athenaire,8,2,10,2,0,zionism vs propalestine antiwar antihate antig...



Missing values:
number          0
link            0
text            0
user            1
likes           0
quotes          0
retweets        0
comments        0
label           0
cleaned_text    1
dtype: int64

Sample row:


number                                                          1
link            https://twitter.com/BabylonBulletin/status/171...
text            🚨 #BREAKING 🚨   A high-ranking member of Israe...
user                                             Babylon Bulletin
likes                                                           2
quotes                                                          0
retweets                                                        0
comments                                                        0
label                                                           0
cleaned_text    breaking highranking member israeli special fo...
Name: 0, dtype: object

In [4]:
# Convert timestamp from Unix to datetime
print("Converting Reddit timestamps...")

# Check if timestamp is in seconds or milliseconds
sample_ts = float(reddit_df['timestamp'].iloc[0])
if sample_ts > 1e12:  # Milliseconds
    reddit_df['timestamp'] = pd.to_datetime(reddit_df['timestamp'].astype(float), unit='ms')
else:  # Seconds
    reddit_df['timestamp'] = pd.to_datetime(reddit_df['timestamp'].astype(float), unit='s')

print(f"Timestamp range: {reddit_df['timestamp'].min()} to {reddit_df['timestamp'].max()}")

# Extract temporal features
reddit_df['year'] = reddit_df['timestamp'].dt.year
reddit_df['month'] = reddit_df['timestamp'].dt.month
reddit_df['day'] = reddit_df['timestamp'].dt.day
reddit_df['hour'] = reddit_df['timestamp'].dt.hour
reddit_df['day_of_week'] = reddit_df['timestamp'].dt.dayofweek  # 0=Monday, 6=Sunday

print(f"\nTemporal features added:")
print(reddit_df[['year', 'month', 'day', 'hour', 'day_of_week']].head())

Converting Reddit timestamps...
Timestamp range: 2023-08-01 14:10:27 to 2023-11-30 22:20:54

Temporal features added:
   year  month  day  hour  day_of_week
0  2023     10   13    18            4
1  2023     10    7    18            5
2  2023     11    2     2            3
3  2023     11   24    14            4
4  2023     10   12    14            3


In [5]:
# Define event start time
event_start = pd.to_datetime(EVENT_START)
print(f"Event start: {event_start}")

# Calculate hours since event
reddit_df['hours_since_event'] = (reddit_df['timestamp'] - event_start).dt.total_seconds() / 3600
reddit_df['hour_bin'] = reddit_df['hours_since_event'].apply(lambda x: int(x) if x >= 0 else -1)

print(f"Hours since event range: {reddit_df['hours_since_event'].min():.1f} to {reddit_df['hours_since_event'].max():.1f}")

# Filter to 48-hour window (or adjust as needed)
# Keep events within the window, filter out outliers
reddit_filtered = reddit_df[(reddit_df['hours_since_event'] >= 0) & 
                            (reddit_df['hours_since_event'] <= TIME_WINDOW_HOURS)].copy()

print(f"\nOriginal Reddit rows: {len(reddit_df)}")
print(f"After temporal filtering: {len(reddit_filtered)}")
print(f"Removed {len(reddit_df) - len(reddit_filtered)} rows outside event window")

# Update DataFrame
reddit_df = reddit_filtered.copy()

# Check distribution by hour
hourly_counts = reddit_df['hour_bin'].value_counts().sort_index()
print(f"\nPosts per hour (first 10 hours):")
print(hourly_counts.head(10))

Event start: 2024-06-27 21:00:00
Hours since event range: -7950.8 to -5038.7

Original Reddit rows: 1000
After temporal filtering: 0
Removed 1000 rows outside event window

Posts per hour (first 10 hours):
Series([], Name: count, dtype: int64)


In [6]:
def clean_reddit_text(text):
    """Clean Reddit comment text"""
    if not isinstance(text, str):
        return ""
    
    # Convert to string and lower case
    text = str(text).lower()
    
    # Remove URLs (keep as placeholder)
    text = re.sub(r'http\S+|www\S+|https\S+', '[URL]', text)
    
    # Remove Reddit-specific formatting
    text = re.sub(r'\[deleted\]', '', text)
    text = re.sub(r'\[removed\]', '', text)
    text = re.sub(r'\/s', '', text)  # Remove /s sarcasm tags
    
    # Remove mentions and subreddit links
    text = re.sub(r'u/\w+', '[USER]', text)
    text = re.sub(r'r/\w+', '[SUBREDDIT]', text)
    
    # Remove special characters but keep basic punctuation
    text = re.sub(r'[^a-zA-Z0-9\s\.\,\?\!\ "\'\']', ' ', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

def tokenize_text(text):
    """Tokenize and remove stopwords"""
    if not text or len(text) < 3:
        return []
    
    try:
        # Get stopwords
        stop_words = set(stopwords.words('english'))
        
        # Tokenize
        tokens = word_tokenize(text)
        
        # Remove stopwords and short words
        tokens = [t for t in tokens if t.lower() not in stop_words and len(t) > 2]
        
        return tokens
    except:
        return []

# Apply cleaning
print("Cleaning Reddit text...")
reddit_df['clean_text'] = reddit_df['text'].apply(clean_reddit_text)
reddit_df['tokens'] = reddit_df['clean_text'].apply(tokenize_text)
reddit_df['word_count'] = reddit_df['tokens'].apply(len)

# Filter out very short comments
reddit_df = reddit_df[reddit_df['word_count'] >= 5].copy()

print(f"\nAfter text cleaning and filtering: {len(reddit_df)} rows")
print(f"Word count distribution:")
print(reddit_df['word_count'].describe())

# Show sample of cleaned text
print(f"\nSample of cleaned text:")
display(reddit_df[['text', 'clean_text', 'word_count']].head(5))

Cleaning Reddit text...

After text cleaning and filtering: 0 rows
Word count distribution:
count       0
unique      0
top       NaN
freq      NaN
Name: word_count, dtype: object

Sample of cleaned text:


,text,clean_text,word_count


In [7]:
def detect_language(text):
    """Detect language of text"""
    if not text or len(text) < 10:
        return 'unknown'
    try:
        return detect(text)
    except:
        return 'unknown'

# Apply language detection (may take time for large datasets)
print("Detecting languages (this may take a moment)...")
# Use a subset if dataset is very large
# reddit_df['language'] = reddit_df['clean_text'].apply(detect_language)
# For large datasets, use this efficient version:
if len(reddit_df) > 10000:
    print("Large dataset detected. Using sample for language detection...")
    # Sample to detect language distribution
    sample = reddit_df['clean_text'].sample(min(1000, len(reddit_df)))
    sample_lang = sample.apply(detect_language)
    lang_dist = sample_lang.value_counts()
    print(f"Language distribution in sample:")
    print(lang_dist)
    
    # If most are English, assume all are English (and filter later)
    if lang_dist.index[0] == 'en' and lang_dist.iloc[0] / len(sample) > 0.9:
        reddit_df['language'] = 'en'
    else:
        # Apply to all
        reddit_df['language'] = reddit_df['clean_text'].apply(detect_language)
else:
    reddit_df['language'] = reddit_df['clean_text'].apply(detect_language)

# Filter to English only
reddit_df = reddit_df[reddit_df['language'] == 'en'].copy()

print(f"\nAfter English filtering: {len(reddit_df)} rows")
print(f"Language distribution:")
print(reddit_df['language'].value_counts())

Detecting languages (this may take a moment)...

After English filtering: 0 rows
Language distribution:
Series([], Name: count, dtype: int64)


In [8]:
# Clean X data
print("=== CLEANING X (TWITTER) DATA ===")

# Check column names and rename if needed
print("Current columns:", x_df.columns.tolist())

# Standardize column names
column_mapping = {
    'id': 'tweet_id',
    'text': 'text',
    'date': 'timestamp',
    'likes': 'like_count',
    'comments': 'reply_count',
    'quotes': 'quote_count',
    'retweets': 'retweet_count',
    'Label': 'engagement_label',
    'label': 'engagement_label',
    'Count': 'count_in_category'
}

# Rename columns that exist
for old, new in column_mapping.items():
    if old in x_df.columns:
        x_df = x_df.rename(columns={old: new})

print(f"Renamed columns: {x_df.columns.tolist()}")

# Handle timestamp
if 'timestamp' in x_df.columns and x_df['timestamp'].notna().any():
    x_df['timestamp'] = pd.to_datetime(x_df['timestamp'])
else:
    # Twitter Snowflake IDs in status URLs contain the tweet creation time.
    status_ids = x_df['link'].astype(str).str.extract(r'/status/(\d+)', expand=False)
    valid_status_ids = status_ids.notna()
    if valid_status_ids.any():
        twitter_epoch_ms = 1288834974657
        x_df['timestamp'] = pd.NaT
        x_df.loc[valid_status_ids, 'timestamp'] = pd.to_datetime(
            ((status_ids.loc[valid_status_ids].astype('int64') // (2 ** 22)) + twitter_epoch_ms),
            unit='ms'
        )
        print(f"Derived timestamps from {valid_status_ids.sum()} Twitter status URLs")
    else:
        raise ValueError("No timestamp column or usable Twitter status URLs found")

print(f"Timestamp range: {x_df['timestamp'].min()} to {x_df['timestamp'].max()}")

# Extract temporal features
x_df['year'] = x_df['timestamp'].dt.year
x_df['month'] = x_df['timestamp'].dt.month
x_df['day'] = x_df['timestamp'].dt.day
x_df['hour'] = x_df['timestamp'].dt.hour

=== CLEANING X (TWITTER) DATA ===
Current columns: ['number', 'link', 'text', 'user', 'likes', 'quotes', 'retweets', 'comments', 'label', 'cleaned_text']
Renamed columns: ['number', 'link', 'text', 'user', 'like_count', 'quote_count', 'retweet_count', 'reply_count', 'engagement_label', 'cleaned_text']
Derived timestamps from 7323 Twitter status URLs
Timestamp range: 2023-10-26 11:46:57.435000 to 2024-09-15 23:59:57.529000


In [9]:
# Calculate hours since event
x_df['hours_since_event'] = (x_df['timestamp'] - event_start).dt.total_seconds() / 3600
x_df['hour_bin'] = x_df['hours_since_event'].apply(lambda x: int(x) if x >= 0 else -1)

print(f"Hours since event range: {x_df['hours_since_event'].min():.1f} to {x_df['hours_since_event'].max():.1f}")

# Filter to 48-hour window
x_filtered = x_df[(x_df['hours_since_event'] >= 0) & 
                  (x_df['hours_since_event'] <= TIME_WINDOW_HOURS)].copy()

print(f"\nOriginal X rows: {len(x_df)}")
print(f"After temporal filtering: {len(x_filtered)}")
print(f"Removed {len(x_df) - len(x_filtered)} rows outside event window")

x_df = x_filtered.copy()

Hours since event range: -5889.2 to 1923.0

Original X rows: 7323
After temporal filtering: 137
Removed 7186 rows outside event window


In [10]:
def clean_x_text(text):
    """Clean X/Twitter text"""
    if not isinstance(text, str):
        return ""
    
    # Convert to string and lower case
    text = str(text).lower()
    
    # Remove URLs (keep placeholder)
    text = re.sub(r'http\S+|www\S+|https\S+', '[URL]', text)
    
    # Remove mentions
    text = re.sub(r'@\w+', '[USER]', text)
    
    # Remove hashtags (keep the word without #)
    text = re.sub(r'#', '', text)
    
    # Remove special characters
    text = re.sub(r'[^a-zA-Z0-9\s\.\,\?\!\ "\'\']', ' ', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

# Apply cleaning
print("Cleaning X text...")
x_df['clean_text'] = x_df['text'].apply(clean_x_text)
x_df['tokens'] = x_df['clean_text'].apply(tokenize_text)
x_df['word_count'] = x_df['tokens'].apply(len)

# Filter short tweets
x_df = x_df[x_df['word_count'] >= 5].copy()

print(f"\nAfter text cleaning: {len(x_df)} rows")
print(f"Word count distribution:")
print(x_df['word_count'].describe())

# Sample of cleaned text
print(f"\nSample of cleaned text:")
display(x_df[['text', 'clean_text', 'word_count']].head(5))

# Language filter for X
print("\nFiltering English tweets...")
if len(x_df) > 10000:
    sample = x_df['clean_text'].sample(min(1000, len(x_df)))
    sample_lang = sample.apply(detect_language)
    lang_dist = sample_lang.value_counts()
    print(f"Language distribution in sample:")
    print(lang_dist)
    
    if lang_dist.index[0] == 'en' and lang_dist.iloc[0] / len(sample) > 0.9:
        x_df['language'] = 'en'
        x_df = x_df[x_df['language'] == 'en']
    else:
        x_df['language'] = x_df['clean_text'].apply(detect_language)
        x_df = x_df[x_df['language'] == 'en']
else:
    x_df['language'] = x_df['clean_text'].apply(detect_language)
    x_df = x_df[x_df['language'] == 'en']

print(f"\nAfter English filtering: {len(x_df)} rows")

Cleaning X text...



After text cleaning: 132 rows
Word count distribution:
count    132.000000
mean      16.575758
std        7.897358
min        5.000000
25%       10.000000
50%       15.000000
75%       23.000000
max       52.000000
Name: word_count, dtype: float64

Sample of cleaned text:


,text,clean_text,word_count
682,#CrimeMinister 🇮🇱 Thousands of Israelis are de...,crimeminister thousands of israelis are demons...,33
683,The #Palestinian Red Crescent Society announce...,the palestinian red crescent society announced...,52
684,#bbtvi #BINI #BINIverse1stSoloConD2 #MilkLoveF...,bbtvi bini biniverse1stsolocond2 milklovefanda...,11
685,Israeli forces targeted a group of #Palestinia...,israeli forces targeted a group of palestinian...,25
686,The #Israeli military has said its troops are ...,the israeli military has said its troops are i...,23



Filtering English tweets...

After English filtering: 128 rows


In [11]:
# Add platform labels
reddit_df['platform'] = 'Reddit'
x_df['platform'] = 'X'

print(f"Reddit: {len(reddit_df)} rows")
print(f"X: {len(x_df)} rows")
print(f"Total combined: {len(reddit_df) + len(x_df)} rows")

Reddit: 0 rows
X: 128 rows
Total combined: 128 rows


In [14]:
# Save processed data
%pip install pyarrow -q

print("Saving processed data...")

# Save Reddit data
reddit_output = DATA_PROCESSED / 'reddit_processed.parquet'
reddit_df.to_parquet(reddit_output, index=False)
print(f" Reddit data saved: {reddit_output}")

# Save X data
x_output = DATA_PROCESSED / 'x_processed.parquet'
x_df.to_parquet(x_output, index=False)
print(f" X data saved: {x_output}")

# Save combined for reference
reddit_df['source'] = 'Reddit'
x_df['source'] = 'X'
combined_df = pd.concat([reddit_df, x_df], ignore_index=True)
combined_output = DATA_PROCESSED / 'combined_processed.parquet'
combined_df.to_parquet(combined_output, index=False)
print(f" Combined data saved: {combined_output}")

# Also save as CSV for easy viewing
reddit_df.to_csv(DATA_PROCESSED / 'reddit_processed.csv', index=False)
x_df.to_csv(DATA_PROCESSED / 'x_processed.csv', index=False)
combined_df.to_csv(DATA_PROCESSED / 'combined_processed.csv', index=False)
print(f" CSV versions also saved")

# Save processing summary
processing_summary = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'reddit_original': len(reddit_data) if 'reddit_data' in locals() else 0,
    'reddit_final': len(reddit_df),
    'x_original': len(x_df) if 'x_df' in locals() else 0,
    'x_final': len(x_df),
    'combined_final': len(combined_df),
    'event_start': str(event_start),
    'time_window_hours': TIME_WINDOW_HOURS
}

import json
summary_file = DATA_PROCESSED / 'processing_summary.json'
with open(summary_file, 'w') as f:
    json.dump(processing_summary, f, indent=2)

print(f"\n Processing summary saved: {summary_file}")
print(f"Summary: {processing_summary}")

Note: you may need to restart the kernel to use updated packages.
Saving processed data...
 Reddit data saved: c:\Users\GALLOP SEA\Desktop\The_Echo_Of_Anonimity-1\src\data\processed\reddit_processed.parquet
 X data saved: c:\Users\GALLOP SEA\Desktop\The_Echo_Of_Anonimity-1\src\data\processed\x_processed.parquet
 Combined data saved: c:\Users\GALLOP SEA\Desktop\The_Echo_Of_Anonimity-1\src\data\processed\combined_processed.parquet
 CSV versions also saved

 Processing summary saved: c:\Users\GALLOP SEA\Desktop\The_Echo_Of_Anonimity-1\src\data\processed\processing_summary.json
Summary: {'timestamp': '2026-08-28 08:51:03', 'reddit_original': 1000, 'reddit_final': 0, 'x_original': 128, 'x_final': 128, 'combined_final': 128, 'event_start': '2024-06-27 21:00:00', 'time_window_hours': 48}



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
# Quick validation
print("=== DATA VALIDATION ===")

# Check for missing values
print("\nMissing values in Reddit data:")
print(reddit_df.isnull().sum()[reddit_df.isnull().sum() > 0])

print("\nMissing values in X data:")
print(x_df.isnull().sum()[x_df.isnull().sum() > 0])

# Check column consistency
common_cols = set(reddit_df.columns) & set(x_df.columns)
print(f"\nCommon columns: {sorted(common_cols)}")

reddit_only = set(reddit_df.columns) - set(x_df.columns)
x_only = set(x_df.columns) - set(reddit_df.columns)
print(f"Reddit-only columns: {reddit_only}")
print(f"X-only columns: {x_only}")

# Check data ranges
print(f"\nReddit timestamp range: {reddit_df['timestamp'].min()} to {reddit_df['timestamp'].max()}")
print(f"X timestamp range: {x_df['timestamp'].min()} to {x_df['timestamp'].max()}")

print("\n✅ Validation complete!")

=== DATA VALIDATION ===

Missing values in Reddit data:
Series([], dtype: int64)

Missing values in X data:
Series([], dtype: int64)

Common columns: ['clean_text', 'day', 'hour', 'hour_bin', 'hours_since_event', 'language', 'month', 'platform', 'source', 'text', 'timestamp', 'tokens', 'word_count', 'year']
Reddit-only columns: {'comment_df', 'num_comments', 'submission_id', 'downs', 'id', 'upvote_ratio', 'datetime', 'author', 'comments_df', 'controversial', 'freepalestine_islamophobia', 'score', 'day_of_week', 'subreddit', 'zionism_antisemitism', 'parent_id', 'title', 'ups', 'upvotes'}
X-only columns: {'number', 'quote_count', 'link', 'user', 'engagement_label', 'like_count', 'cleaned_text', 'retweet_count', 'reply_count'}

Reddit timestamp range: NaT to NaT
X timestamp range: 2024-06-29 20:18:17.823000 to 2024-06-29 20:59:54.949000

✅ Validation complete!
